In [ ]:
import pandas as pd
import json
import os
import re
from pathlib import Path
from tqdm import tqdm

# 데이터 경로 설정
base_path = Path(".")
training_path = base_path / "Training" / "02.라벨링데이터"
validation_path = base_path / "Validation" / "02.라벨링데이터"

def clean_text(text, role):
    """
    텍스트 전처리: 공백/개행 제거, 지칭 변환
    """
    # strip
    text = text.strip()
    
    # 감정화자 → A, 공감화자 → B로 변환
    text = re.sub(r'감정화자(님)?', 'A', text)
    text = re.sub(r'공감화자(님)?', 'B', text)
    
    # 화자님, 청자님 등도 처리
    if role == 'speaker':
        text = re.sub(r'화자(님)?', 'A', text)
        text = re.sub(r'청자(님)?', 'B', text)
    else:  # listener
        text = re.sub(r'화자(님)?', 'A', text)
        text = re.sub(r'청자(님)?', 'B', text)
    
    return text

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_json(json_file, data_type, empathy_classes):
    """단일 JSON 파일 처리"""
    try:
        file_name = json_file.stem
        parts = file_name.split('_')
        
        if len(parts) >= 3:
            emotion = parts[1]
            conv_id = parts[-1]
            relation = '_'.join(parts[2:-1])
        else:
            emotion = relation = conv_id = None
        
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        info = data['info']
        rows = []
        
        for idx, utterance in enumerate(data['utterances']):
            empathy_ohe = {f'empathy_{cls}': 0 for cls in empathy_classes}
            if utterance.get('listener_empathy'):
                for emp in utterance['listener_empathy']:
                    if emp in empathy_classes:
                        empathy_ohe[f'empathy_{emp}'] = 1
            
            role = utterance['role']
            
            row = {
                'data_type': data_type,
                'emotion': emotion,
                'relation': relation,
                'conv_id': conv_id,
                'utterance_num': idx + 1,
                'situation': info['situation'].strip(),
                'avg_rating': info['evaluation']['avg_rating'],
                'grade': info['evaluation']['grade'],
                'role': role,
                'text': clean_text(utterance['text'], role),
                'terminate': utterance['terminate'],
                **empathy_ohe,
                'speaker_changeEmotion': utterance.get('speaker_changeEmotion')
            }
            rows.append(row)
        
        return rows
    except Exception as e:
        print(f"Error processing {json_file}: {e}")
        return []


def load_json_to_dataframe(data_path, data_type='train', relations=None, max_workers=8):
    """
    JSON 파일들을 병렬로 읽어서 DataFrame으로 변환
    
    Args:
        data_path: JSON 파일들이 있는 폴더 경로
        data_type: 'train' 또는 'validation'
        relations: 필터링할 관계 리스트 (예: ['친구', '지인']). None이면 전체
        max_workers: 병렬 처리 스레드 수
    
    Returns:
        pandas.DataFrame
    """
    empathy_classes = ['위로', '격려', '조언', '동조']
    
    # 하위 폴더 탐색 (relations 필터 적용)
    if relations:
        folders = [f for f in data_path.iterdir() if f.is_dir() and any(r in f.name for r in relations)]
    else:
        folders = [f for f in data_path.iterdir() if f.is_dir()]
    
    # 모든 JSON 파일 수집
    all_json_files = []
    for folder in folders:
        all_json_files.extend(list(folder.glob("*.json")))
    
    print(f"Processing {len(all_json_files)} files from {len(folders)} folders ({data_type})...")
    
    # 병렬 처리
    all_data = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_single_json, f, data_type, empathy_classes): f 
            for f in all_json_files
        }
        
        for future in tqdm(as_completed(futures), total=len(futures)):
            rows = future.result()
            all_data.extend(rows)
    
    # DataFrame 생성
    df = pd.DataFrame(all_data)
    
    column_order = [
        'data_type', 'emotion', 'relation', 'conv_id', 'utterance_num',
        'situation', 'avg_rating', 'grade',
        'role', 'text', 
        'empathy_위로', 'empathy_격려', 'empathy_조언', 'empathy_동조',
        'speaker_changeEmotion', 'terminate'
    ]
    
    df = df[column_order]
    
    return df

In [ ]:
# 친구, 지인 관계만 필터링해서 로드
target_relations = ['친구', '지인']

train_df = load_json_to_dataframe(training_path, data_type='train', relations=target_relations)
valid_df = load_json_to_dataframe(validation_path, data_type='validation', relations=target_relations)

print(f"\nTrain: {len(train_df):,} rows")
print(f"Valid: {len(valid_df):,} rows")
print(f"Total: {len(train_df) + len(valid_df):,} rows")

In [ ]:
# train_df와 valid_df를 합치고 text 평균 길이 계산
merged_df = pd.concat([train_df, valid_df], ignore_index=True)

# text 길이 계산
merged_df['text_len'] = merged_df['text'].str.len()

print(f"=== Merged DataFrame ===")
print(f"총 행 수: {len(merged_df):,}개")
print(f"\n=== Text 평균 길이 ===")
print(f"평균: {merged_df['text_len'].mean():.2f}자")
print(f"중앙값: {merged_df['text_len'].median():.1f}자")
print(f"최소: {merged_df['text_len'].min()}자")
print(f"최대: {merged_df['text_len'].max()}자")
print(f"표준편차: {merged_df['text_len'].std():.2f}자")

In [ ]:
# Role별 text 평균 길이
print("=== Role별 Text 길이 통계 ===")
role_stats = merged_df.groupby('role')['text_len'].agg(['mean', 'median', 'min', 'max', 'std', 'count'])
role_stats.columns = ['평균', '중앙값', '최소', '최대', '표준편차', '개수']
print(role_stats.round(2))

print("\n=== Role별 상세 ===")
for role in merged_df['role'].unique():
    role_data = merged_df[merged_df['role'] == role]['text_len']
    print(f"\n[{role}]")
    print(f"  평균: {role_data.mean():.2f}자")
    print(f"  중앙값: {role_data.median():.1f}자")
    print(f"  최소: {role_data.min()}자 / 최대: {role_data.max()}자")
    print(f"  개수: {len(role_data):,}개")

In [ ]:
# ========================================
# Role별 Text 길이 시각화 (Histogram)
# ========================================
import matplotlib.pyplot as plt
import seaborn as sns

# Role 영문 매핑
role_map = {'speaker': 'Speaker', 'listener': 'Listener'}
merged_df['role_en'] = merged_df['role'].map(role_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, role in enumerate(['Speaker', 'Listener']):
    ax = axes[idx]
    subset = merged_df[merged_df['role_en'] == role]['text_len']
    ax.hist(subset, bins=50, color='steelblue' if role == 'Speaker' else 'coral', edgecolor='white')
    ax.set_title(f'{role} Text Length', fontsize=12, fontweight='bold')
    ax.set_xlabel('Text Length (chars)')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Role + Emotion별 Text 길이 Histogram
# ========================================

# Emotion 영문 매핑
emotion_map = {
    '기쁨': 'Joy',
    '슬픔': 'Sadness', 
    '분노': 'Anger',
    '불안': 'Anxiety',
    '상처': 'Hurt',
    '당황': 'Embarrassment'
}
merged_df['emotion_en'] = merged_df['emotion'].map(emotion_map)

emotions = merged_df['emotion_en'].dropna().unique().tolist()
roles = ['Speaker', 'Listener']
colors = {'Speaker': 'steelblue', 'Listener': 'coral'}

fig, axes = plt.subplots(len(emotions), 2, figsize=(14, 4 * len(emotions)))

for row_idx, emotion in enumerate(emotions):
    for col_idx, role in enumerate(roles):
        ax = axes[row_idx, col_idx]
        subset = merged_df[(merged_df['emotion_en'] == emotion) & (merged_df['role_en'] == role)]['text_len']
        
        # 히스토그램
        ax.hist(subset, bins=50, color=colors[role], edgecolor='white', alpha=0.7)
        
        # Mean, Median 선
        mean_val = subset.mean()
        median_val = subset.median()
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
        ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
        
        ax.set_title(f'{emotion} - {role}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Text Length (chars)')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8)

plt.suptitle('Text Length Distribution by Emotion & Role', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [8]:
train_df.head(3).T

,0,1,2
data_type,train,train,train
emotion,기쁨,기쁨,기쁨
relation,부모자녀_조손,부모자녀_조손,부모자녀_조손
conv_id,1,1,1
utterance_num,1,2,3
situation,아이가 태어나니 부모가 배우고 익혀야 할 것이 많다.,아이가 태어나니 부모가 배우고 익혀야 할 것이 많다.,아이가 태어나니 부모가 배우고 익혀야 할 것이 많다.
avg_rating,5.0,5.0,5.0
grade,우수,우수,우수
role,speaker,listener,speaker
text,"엄마, 아기가 태어나니까 내가 부모로서 해야 할 게 참 많은 것 같아요.",그렇지? 아기 키우는 게 여간 어려운 일이 아니야.,어제 평소보다도 격하게 막 온몸을 써가면서 울더라고요. 얼마나 당황했는지 몰라요.


In [11]:
listener_df = train_df[train_df['role'] == 'listener']
no_empathy = listener_df[
    (listener_df['empathy_위로'] == 0) & 
    (listener_df['empathy_격려'] == 0) & 
    (listener_df['empathy_조언'] == 0) & 
    (listener_df['empathy_동조'] == 0)
]

In [13]:
no_empathy.text

3                   배가 고파서 그랬던 것 아닐까? 아기들은 배가 고프면 몸부림을 친단다.
5                 말도 마. 네가 얼마나 까탈스러웠는데. 우리 손주가 내 딸을 빼다 박았네.
16                                    이렇게 갑자기? 누구랑 같이 간 거야?
33                얘는 뜬금없이 무슨 소리를 하는 거야? 그런 건 엄마한테 직접 물어봐야지.
35                     글쎄, 뭐라고 했길래 우리 딸 기분에 봄바람이 살랑살랑 불어올까?
                                ...                        
378528                                  그래, 언제나 이 언니가 응원할게.
378540         형부가 조금 분석적이고 날카로워서 마음 약한 언니가 자주 상처를 받는 것 같아.
378547                                       우리 동생 무슨 일 있어?
378555    그 친구도 우리 동생이랑 친하게 지내고 싶어서 더 못되게 굴었나보다. 그렇다고 그 ...
378557                                   그랬더니 친구가 뭐라고 대답했어?
Name: text, Length: 36446, dtype: object

In [15]:
listener_df.text

1                              그렇지? 아기 키우는 게 여간 어려운 일이 아니야.
3                   배가 고파서 그랬던 것 아닐까? 아기들은 배가 고프면 몸부림을 친단다.
5                 말도 마. 네가 얼마나 까탈스러웠는데. 우리 손주가 내 딸을 빼다 박았네.
7                          네가 어른 노릇, 부모 노릇을 하느라 고생이 정말 많구나.
9         우리 딸이 엄마가 되더니 철이 들었네. 하지만 네가 웃어주면 세상 모든 시름이 사라...
                                ...                        
378553    나라도 누군가가 나에게 무작정 강요했으면 기분이 시궁창으로 처박히는 느낌이었을 거야...
378555    그 친구도 우리 동생이랑 친하게 지내고 싶어서 더 못되게 굴었나보다. 그렇다고 그 ...
378557                                   그랬더니 친구가 뭐라고 대답했어?
378559    친구가 그래도 잘못을 스스로 깨달은 것 같은데, 한 번 더 기회를 줘보는 건 어떨까...
378561    그러면 조금 더 지내보고 마음을 결정해도 될 것 같아. 혹시나 또 상처받을 일이 있...
Name: text, Length: 181802, dtype: object

In [20]:
train_df.relation.value_counts()

친구         58290
형제_자매      54055
연인         53748
직장 동료      53528
지인         53410
부모자녀_조손    52887
부부         52645
Name: relation, dtype: int64

In [22]:
train_df[train_df.relation == '지인'].text

34798                               우리 아파트 계단이 요즘 너무 깨끗해졌어.
34799     아 진짜? 너 지난번에 너희 아파트 계단에 먼지도 많고 담배꽁초도 버려져 있다고 보...
34800     그랬었지. 근데 이번에 새로 청소하시는 아주머니께서 오셨는데 너무 깔끔하고 꼼꼼하게...
34801                       너무 좋은 일이다. 아파트가 머지않아 반짝반짝해지겠는걸?
34802     그러니까 옆집 주민도 나와 같은 생각인가 봐. 우연히 만났는데 청소 아주머니를 막 ...
                                ...                        
349413              정말 답답하죠. 몇 억을 들여서라도 좋은 피부를 얻고 싶은 심정이에요.
349414    그 마음 충분히 공감합니다. 앞으로도 언제든 슬프고 우울할 때 연락을 주세요. 기댈...
349415              그렇게 말씀해주시니 정말 감사합니다. 제 이야기도 들어주시고 말이에요.
349416    저도 선생님에게 긍정적인 기운을 많이 받았었잖아요. 들어드리는 것은 언제든 할 수 ...
349417                   감사합니다. 덕분에 우울했던 기분이 한층 나아지는 것 같아요.
Name: text, Length: 53410, dtype: object

In [ ]:
def create_single_turn_qa(df, stride=1):
    """
    연속된 2개 발화를 Q → A 형태로 변환 (싱글턴 QA)
    
    Args:
        df: 발화 DataFrame
        stride: 슬라이딩 간격 (기본 1)
    
    Returns:
        DataFrame
    """
    qa_data = []
    
    df = df.sort_values(['emotion', 'relation', 'conv_id', 'utterance_num'])
    grouped = df.groupby(['data_type', 'emotion', 'relation', 'conv_id'])
    
    for (data_type, emotion, relation, conv_id), group in tqdm(grouped, desc="Creating single-turn QA"):
        utterances = group.to_dict('records')
        
        for i in range(0, len(utterances) - 1, stride):
            query_turn = utterances[i]
            response_turn = utterances[i + 1]
            
            qa_data.append({
                'data_type': data_type,
                'emotion': emotion,
                'relation': relation,
                'conv_id': conv_id,
                'situation': utterances[0]['situation'],
                'query': query_turn['text'],
                'query_role': 'A' if query_turn['role'] == 'speaker' else 'B',
                'response': response_turn['text'],
                'response_role': 'A' if response_turn['role'] == 'speaker' else 'B',
                'empathy_위로': response_turn['empathy_위로'],
                'empathy_격려': response_turn['empathy_격려'],
                'empathy_조언': response_turn['empathy_조언'],
                'empathy_동조': response_turn['empathy_동조'],
                'speaker_changeEmotion': response_turn['speaker_changeEmotion']
            })
    
    return pd.DataFrame(qa_data)


def create_multi_turn_qa(df, window_size=3, stride=1):
    """
    대화를 슬라이딩 윈도우로 나눠서 context → response 형태로 변환 (멀티턴 QA)
    
    Args:
        df: 발화 DataFrame
        window_size: 윈도우 크기 (기본 3턴: 2턴 context + 1턴 response)
        stride: 슬라이딩 간격 (기본 1)
    
    Returns:
        DataFrame
    """
    qa_data = []
    
    df = df.sort_values(['emotion', 'relation', 'conv_id', 'utterance_num'])
    grouped = df.groupby(['data_type', 'emotion', 'relation', 'conv_id'])
    
    for (data_type, emotion, relation, conv_id), group in tqdm(grouped, desc="Creating multi-turn QA"):
        utterances = group.to_dict('records')
        
        if len(utterances) < window_size:
            continue
        
        for i in range(0, len(utterances) - window_size + 1, stride):
            window = utterances[i:i + window_size]
            
            context_turns = window[:-1]
            response_turn = window[-1]
            
            context_lines = []
            for turn in context_turns:
                role_label = "A" if turn['role'] == 'speaker' else "B"
                context_lines.append(f"{role_label}: {turn['text']}")
            
            context = "\n".join(context_lines)
            response_role = "A" if response_turn['role'] == 'speaker' else "B"
            
            qa_data.append({
                'data_type': data_type,
                'emotion': emotion,
                'relation': relation,
                'conv_id': conv_id,
                'situation': utterances[0]['situation'],
                'context': context,
                'response': response_turn['text'],
                'response_role': response_role,
                'empathy_위로': response_turn['empathy_위로'],
                'empathy_격려': response_turn['empathy_격려'],
                'empathy_조언': response_turn['empathy_조언'],
                'empathy_동조': response_turn['empathy_동조'],
                'speaker_changeEmotion': response_turn['speaker_changeEmotion']
            })
    
    return pd.DataFrame(qa_data)

In [ ]:
# 싱글턴 QA (stride=1, 모든 연속 발화쌍)
single_qa_df = create_single_turn_qa(train_df, stride=1)
print(f"Single-turn QA: {len(single_qa_df):,}")

# 멀티턴 QA (window=3, stride=1)
multi_qa_df = create_multi_turn_qa(train_df, window_size=3, stride=1)
print(f"Multi-turn QA: {len(multi_qa_df):,}")

In [ ]:
# 싱글턴 예시 확인
print("=== Single-turn QA 예시 ===")
sample = single_qa_df.iloc[0]
print(f"[상황] {sample['situation']}")
print(f"[Q] {sample['query']}")
print(f"[A] {sample['response']}")
print(f"[공감] 위로:{sample['empathy_위로']} 격려:{sample['empathy_격려']} 조언:{sample['empathy_조언']} 동조:{sample['empathy_동조']}")

In [ ]:
# 멀티턴 예시 확인
print("=== Multi-turn QA 예시 ===")
sample = multi_qa_df.iloc[0]
print(f"[상황] {sample['situation']}")
print(f"[Context]\n{sample['context']}")
print(f"[Response] {sample['response']}")
print(f"[공감] 위로:{sample['empathy_위로']} 격려:{sample['empathy_격려']} 조언:{sample['empathy_조언']} 동조:{sample['empathy_동조']}")

In [ ]:
# 포맷 통일 후 합치기
def format_for_training(single_df, multi_df):
    """싱글턴/멀티턴을 통일된 형식으로 변환 후 합침"""
    
    formatted = []
    
    # 싱글턴: query → context로 변환
    for _, row in tqdm(single_df.iterrows(), total=len(single_df), desc="Formatting single-turn"):
        formatted.append({
            'type': 'single',
            'emotion': row['emotion'],
            'relation': row['relation'],
            'conv_id': row['conv_id'],
            'instruction': f"[상황] {row['situation']}\n\n[대화]\n{row['query_role']}: {row['query']}",
            'output': row['response'],
            'empathy_위로': row['empathy_위로'],
            'empathy_격려': row['empathy_격려'],
            'empathy_조언': row['empathy_조언'],
            'empathy_동조': row['empathy_동조'],
        })
    
    # 멀티턴
    for _, row in tqdm(multi_df.iterrows(), total=len(multi_df), desc="Formatting multi-turn"):
        formatted.append({
            'type': 'multi',
            'emotion': row['emotion'],
            'relation': row['relation'],
            'conv_id': row['conv_id'],
            'instruction': f"[상황] {row['situation']}\n\n[대화]\n{row['context']}",
            'output': row['response'],
            'empathy_위로': row['empathy_위로'],
            'empathy_격려': row['empathy_격려'],
            'empathy_조언': row['empathy_조언'],
            'empathy_동조': row['empathy_동조'],
        })
    
    return pd.DataFrame(formatted)

# 합치기
combined_df = format_for_training(single_qa_df, multi_qa_df)
print(f"Combined dataset: {len(combined_df):,}")
print(f"- Single: {len(combined_df[combined_df['type']=='single']):,}")
print(f"- Multi: {len(combined_df[combined_df['type']=='multi']):,}")

In [ ]:
# 예시 확인
print("=== 싱글턴 예시 ===")
sample = combined_df[combined_df['type']=='single'].iloc[0]
print(sample['instruction'])
print(f"\n[응답] {sample['output']}")

print("\n" + "="*50)
print("\n=== 멀티턴 예시 ===")
sample = combined_df[combined_df['type']=='multi'].iloc[0]
print(sample['instruction'])
print(f"\n[응답] {sample['output']}")

In [ ]:
# Speaker 발화 분석
speaker_df = train_df[train_df['role'] == 'speaker']

print("=== Speaker 발화 분석 ===")
print(f"총 speaker 발화: {len(speaker_df):,}")

# empathy 태그 있는 speaker?
has_empathy = speaker_df[
    (speaker_df['empathy_위로'] == 1) | 
    (speaker_df['empathy_격려'] == 1) | 
    (speaker_df['empathy_조언'] == 1) | 
    (speaker_df['empathy_동조'] == 1)
]
print(f"empathy 태그 있는 speaker: {len(has_empathy):,}")

# speaker_changeEmotion 분포
print(f"\nspeaker_changeEmotion 분포:")
print(speaker_df['speaker_changeEmotion'].value_counts(dropna=False))

In [ ]:
# ========================================
# 1. 친구/지인 대화 샘플 수 확인 (train/valid)
# ========================================

# 대화 단위로 unique 개수 세기
def count_conversations(df):
    return df.groupby(['relation', 'emotion', 'conv_id']).ngroups

print("=== 친구 대화 샘플 수 ===")
friend_train = train_df[train_df['relation'] == '친구']
friend_valid = valid_df[valid_df['relation'] == '친구']
print(f"Train: {friend_train.groupby(['emotion', 'conv_id']).ngroups:,}개")
print(f"Valid: {friend_valid.groupby(['emotion', 'conv_id']).ngroups:,}개")

print("\n=== 지인 대화 샘플 수 ===")
acquaint_train = train_df[train_df['relation'] == '지인']
acquaint_valid = valid_df[valid_df['relation'] == '지인']
print(f"Train: {acquaint_train.groupby(['emotion', 'conv_id']).ngroups:,}개")
print(f"Valid: {acquaint_valid.groupby(['emotion', 'conv_id']).ngroups:,}개")

print("\n=== 전체 (친구+지인) ===")
print(f"Train: {train_df.groupby(['relation', 'emotion', 'conv_id']).ngroups:,}개")
print(f"Valid: {valid_df.groupby(['relation', 'emotion', 'conv_id']).ngroups:,}개")

In [ ]:
# ========================================
# Train + Valid 합쳐서 QA 데이터 생성 (Listener만)
# ========================================

full_df = pd.concat([train_df, valid_df], ignore_index=True)
print(f"전체 발화: {len(full_df):,}개")

# 싱글턴 + 멀티턴 QA 생성
single_qa_full = create_single_turn_qa(full_df, stride=1)
multi_qa_full = create_multi_turn_qa(full_df, window_size=3, stride=1)

# Listener(B)만 필터링
single_qa_listener = single_qa_full[single_qa_full['response_role'] == 'B']
multi_qa_listener = multi_qa_full[multi_qa_full['response_role'] == 'B']

print(f"\n=== Listener만 필터링 후 ===")
print(f"Single-turn QA: {len(single_qa_listener):,}개")
print(f"Multi-turn QA: {len(multi_qa_listener):,}개")

# Train/Valid 분리
single_train = single_qa_listener[single_qa_listener['data_type'] == 'train']
single_valid = single_qa_listener[single_qa_listener['data_type'] == 'validation']
multi_train = multi_qa_listener[multi_qa_listener['data_type'] == 'train']
multi_valid = multi_qa_listener[multi_qa_listener['data_type'] == 'validation']

print(f"\n=== Train/Valid 분포 ===")
print(f"Single-turn Train: {len(single_train):,}개")
print(f"Single-turn Valid: {len(single_valid):,}개")
print(f"Multi-turn Train: {len(multi_train):,}개")
print(f"Multi-turn Valid: {len(multi_valid):,}개")

In [ ]:
# ========================================
# empathy_train_qa, empathy_valid_qa 기준 통계
# ========================================

print("=== QA 데이터셋 크기 ===")
print(f"Train: {len(empathy_train_qa):,}개")
print(f"Valid: {len(empathy_valid_qa):,}개")
print(f"Total: {len(empathy_train_qa) + len(empathy_valid_qa):,}개")

# 관계별 분포
print("\n=== 관계별 분포 ===")
print("Train:")
print(empathy_train_qa['relation'].value_counts())
print("\nValid:")
print(empathy_valid_qa['relation'].value_counts())

# 감정별 분포
print("\n=== 감정별 분포 ===")
print("Train:")
print(empathy_train_qa['emotion'].value_counts())

In [ ]:
# ========================================
# Context, Situation, Response 길이 통계
# ========================================

# context 컬럼이 있는지 확인 (멀티턴만 있음)
if 'context' in empathy_train_qa.columns:
    context_lens = empathy_train_qa['context'].dropna().str.len()
    print("=== Context 길이 (글자 수) ===")
    print(f"평균: {context_lens.mean():.1f}자")
    print(f"최소: {context_lens.min()}자")
    print(f"최대: {context_lens.max()}자")

print("\n=== Situation 길이 (글자 수) ===")
situation_lens = empathy_train_qa['situation'].str.len()
print(f"평균: {situation_lens.mean():.1f}자")
print(f"최소: {situation_lens.min()}자")
print(f"최대: {situation_lens.max()}자")

print("\n=== Response 길이 (글자 수) ===")
response_lens = empathy_train_qa['response'].str.len()
print(f"평균: {response_lens.mean():.1f}자")
print(f"최소: {response_lens.min()}자")
print(f"최대: {response_lens.max()}자")

# 대화당 평균 청크 수
print("\n=== 대화당 청크 수 ===")
chunks_per_conv = empathy_train_qa.groupby(['relation', 'emotion', 'conv_id']).size()
print(f"평균: {chunks_per_conv.mean():.1f}개")
print(f"최소: {chunks_per_conv.min()}개")
print(f"최대: {chunks_per_conv.max()}개")

In [ ]:
# ========================================
# relation, emotion, data_type별 길이 통계
# ========================================

# 전체 QA 합치기
full_qa = pd.concat([empathy_train_qa, empathy_valid_qa], ignore_index=True)

# 길이 컬럼 추가
full_qa['situation_len'] = full_qa['situation'].str.len()
full_qa['response_len'] = full_qa['response'].str.len()
if 'context' in full_qa.columns:
    full_qa['context_len'] = full_qa['context'].str.len()

print("=== Relation별 길이 통계 ===")
print(full_qa.groupby('relation')[['situation_len', 'response_len']].agg(['mean', 'min', 'max']).round(1))

print("\n=== Emotion별 길이 통계 ===")
print(full_qa.groupby('emotion')[['situation_len', 'response_len']].agg(['mean', 'min', 'max']).round(1))

print("\n=== Data Type별 길이 통계 ===")
print(full_qa.groupby('data_type')[['situation_len', 'response_len']].agg(['mean', 'min', 'max']).round(1))

In [ ]:
# ========================================
# Emotion별 Context/Response 길이 상세 통계
# ========================================
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 → 영문 매핑
emotion_map = {
    '기쁨': 'Joy',
    '슬픔': 'Sadness', 
    '분노': 'Anger',
    '불안': 'Anxiety',
    '상처': 'Hurt',
    '당황': 'Embarrassment'
}

# emotion 영문 컬럼 추가
full_qa['emotion_en'] = full_qa['emotion'].map(emotion_map)

# context_len이 없으면 추가
if 'context_len' not in full_qa.columns and 'context' in full_qa.columns:
    full_qa['context_len'] = full_qa['context'].str.len()

# Emotion별 통계 테이블
print("=== Emotion별 Context 길이 통계 ===")
if 'context_len' in full_qa.columns:
    context_stats = full_qa.groupby('emotion_en')['context_len'].agg(['mean', 'median', 'min', 'max', 'std', 'count'])
    context_stats.columns = ['Mean', 'Median', 'Min', 'Max', 'Std', 'Count']
    print(context_stats.round(2))

print("\n=== Emotion별 Response 길이 통계 ===")
response_stats = full_qa.groupby('emotion_en')['response_len'].agg(['mean', 'median', 'min', 'max', 'std', 'count'])
response_stats.columns = ['Mean', 'Median', 'Min', 'Max', 'Std', 'Count']
print(response_stats.round(2))

In [ ]:
# ========================================
# Emotion별 Response 길이 Histogram (개별) + Mean/Median 선
# ========================================

emotions = full_qa['emotion_en'].dropna().unique().tolist()
n_emotions = len(emotions)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = ['steelblue', 'coral', 'forestgreen', 'purple', 'orange', 'crimson']

for idx, emotion in enumerate(emotions):
    ax = axes[idx]
    subset = full_qa[full_qa['emotion_en'] == emotion]['response_len']
    
    # 히스토그램
    ax.hist(subset, bins=50, color=colors[idx % len(colors)], edgecolor='white', alpha=0.7)
    
    # Mean, Median 선
    mean_val = subset.mean()
    median_val = subset.median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
    
    ax.set_title(f'{emotion}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Response Length (chars)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Response Length Distribution by Emotion', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Emotion별 Context 길이 Histogram (개별) + Mean/Median 선
# ========================================

if 'context_len' in full_qa.columns:
    context_data = full_qa[full_qa['context_len'].notna()]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    colors = ['steelblue', 'coral', 'forestgreen', 'purple', 'orange', 'crimson']
    
    for idx, emotion in enumerate(emotions):
        ax = axes[idx]
        subset = context_data[context_data['emotion_en'] == emotion]['context_len']
        
        # 히스토그램
        ax.hist(subset, bins=50, color=colors[idx % len(colors)], edgecolor='white', alpha=0.7)
        
        # Mean, Median 선
        mean_val = subset.mean()
        median_val = subset.median()
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
        ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
        
        ax.set_title(f'{emotion}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Context Length (chars)')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8)
    
    plt.suptitle('Context Length Distribution by Emotion', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No context data available")

In [ ]:
# ========================================
# Empathy 태그별 Response 길이 Histogram (Listener만)
# ========================================

empathy_map = {
    'empathy_위로': 'Comfort',
    'empathy_격려': 'Encouragement',
    'empathy_조언': 'Advice',
    'empathy_동조': 'Agreement'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = ['steelblue', 'coral', 'forestgreen', 'purple']

for idx, (col, label) in enumerate(empathy_map.items()):
    ax = axes[idx]
    # 해당 empathy 태그가 1인 행만 필터링
    subset = full_qa[full_qa[col] == 1]['response_len']
    
    # 히스토그램
    ax.hist(subset, bins=50, color=colors[idx], edgecolor='white', alpha=0.7)
    
    # Mean, Median 선
    mean_val = subset.mean()
    median_val = subset.median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
    
    ax.set_title(f'{label} (n={len(subset):,})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Response Length (chars)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Response Length by Empathy Type (Listener)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# 대화당 발화 수 분석 (Emotion, Relation별)
# ========================================

# 대화당 발화 수 계산
conv_turns = merged_df.groupby(['emotion', 'relation', 'conv_id']).size().reset_index(name='num_turns')

# 영문 매핑
emotion_map = {
    '기쁨': 'Joy', '슬픔': 'Sadness', '분노': 'Anger',
    '불안': 'Anxiety', '상처': 'Hurt', '당황': 'Embarrassment'
}
relation_map = {
    '친구': 'Friend', '지인': 'Acquaintance'
}
conv_turns['emotion_en'] = conv_turns['emotion'].map(emotion_map)
conv_turns['relation_en'] = conv_turns['relation'].map(relation_map)

# 전체 통계
print("=== Overall Turns per Conversation ===")
print(f"Mean: {conv_turns['num_turns'].mean():.2f}")
print(f"Median: {conv_turns['num_turns'].median():.1f}")
print(f"Min: {conv_turns['num_turns'].min()}, Max: {conv_turns['num_turns'].max()}")
print(f"Total conversations: {len(conv_turns):,}")

In [ ]:
# ========================================
# Emotion별 대화당 발화 수 Histogram
# ========================================

emotions = conv_turns['emotion_en'].dropna().unique().tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = ['steelblue', 'coral', 'forestgreen', 'purple', 'orange', 'crimson']

for idx, emotion in enumerate(emotions):
    ax = axes[idx]
    subset = conv_turns[conv_turns['emotion_en'] == emotion]['num_turns']
    
    ax.hist(subset, bins=30, color=colors[idx], edgecolor='white', alpha=0.7)
    
    mean_val = subset.mean()
    median_val = subset.median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
    
    ax.set_title(f'{emotion}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Turns per Conversation')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Turns per Conversation by Emotion', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Relation별 대화당 발화 수 Histogram
# ========================================

relations = conv_turns['relation_en'].dropna().unique().tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['steelblue', 'coral']

for idx, relation in enumerate(relations):
    ax = axes[idx]
    subset = conv_turns[conv_turns['relation_en'] == relation]['num_turns']
    
    ax.hist(subset, bins=30, color=colors[idx], edgecolor='white', alpha=0.7)
    
    mean_val = subset.mean()
    median_val = subset.median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
    
    ax.set_title(f'{relation}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Turns per Conversation')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Turns per Conversation by Relation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Emotion별 길이 분포 Violin Plot + Histogram
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Response 길이 Violin Plot
ax1 = axes[0]
sns.violinplot(data=full_qa, x='emotion_en', y='response_len', order=emotion_order, ax=ax1, palette='husl')
ax1.set_title('Response Length Distribution (Violin Plot)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Emotion')
ax1.set_ylabel('Response Length (chars)')
ax1.tick_params(axis='x', rotation=45)

# 2. Response 길이 Histogram (감정별 색상)
ax2 = axes[1]
for emotion in emotion_order:
    subset = full_qa[full_qa['emotion_en'] == emotion]['response_len']
    ax2.hist(subset, bins=30, alpha=0.5, label=emotion)
ax2.set_title('Response Length Histogram by Emotion', fontsize=12, fontweight='bold')
ax2.set_xlabel('Response Length (chars)')
ax2.set_ylabel('Frequency')
ax2.legend(title='Emotion')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Context 길이도 포함한 상세 통계 (멀티턴 데이터만)
# ========================================

if 'context_len' in full_qa.columns:
    multi_qa = full_qa[full_qa['context_len'].notna()]
    
    print("=== [멀티턴] Relation별 Context 길이 ===")
    print(multi_qa.groupby('relation')['context_len'].agg(['mean', 'min', 'max']).round(1))
    
    print("\n=== [멀티턴] Emotion별 Context 길이 ===")
    print(multi_qa.groupby('emotion')['context_len'].agg(['mean', 'min', 'max']).round(1))
    
    print("\n=== [멀티턴] Data Type별 Context 길이 ===")
    print(multi_qa.groupby('data_type')['context_len'].agg(['mean', 'min', 'max']).round(1))

In [ ]:
# ========================================
# Empathy 태그별 통계
# ========================================

empathy_cols = ['empathy_위로', 'empathy_격려', 'empathy_조언', 'empathy_동조']
empathy_labels = ['위로', '격려', '조언', '동조']

print("=== Empathy 태그별 Response 길이 통계 ===\n")
for col, label in zip(empathy_cols, empathy_labels):
    with_tag = full_qa[full_qa[col] == 1]['response_len']
    without_tag = full_qa[full_qa[col] == 0]['response_len']
    print(f"[{label}]")
    print(f"  태그 O: 평균 {with_tag.mean():.1f}자, min {with_tag.min()}, max {with_tag.max()}, 개수 {len(with_tag):,}개")
    print(f"  태그 X: 평균 {without_tag.mean():.1f}자, min {without_tag.min()}, max {without_tag.max()}, 개수 {len(without_tag):,}개\n")